# Design Spaces 

## Global Setup

### Import Packages and Directories

In [1]:
import flowtangent as ft
import jax
import jax.numpy as jnp
import numpy as np
import string

from pathlib import Path
from functools import lru_cache

from flowtangent.models.kernels import RBFKernel, Matern52Kernel
from flowtangent.models.surrogates import SVGPR
from flowtangent.models.manifolds import LinearPCA, Passthrough, ScalingManifold
from flowtangent.models.design import ActiveSpace, Parameter, Requirement

from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances_argmin

import jax.numpy as jnp
import equinox as eqx
import optax

@lru_cache(maxsize=2048)
def surf_lookup(name: str):
    safe_name = name.replace("-", "_").replace(".", "_")
    if safe_name and safe_name[0] in string.digits:
        safe_name = f"af_{safe_name}"

    af = getattr(ft.airfoils, safe_name)

    y_af = jnp.concatenate([af.y_upper, af.y_lower])
    return y_af

def name2surf(batch: dict):
    names = batch["foil_name"]
    
    # 1. Strip Xarray metadata if it slipped through
    if hasattr(names, "values"):
        names = names.values
        
    # 2. Force it into a flat, pure NumPy array
    names = np.asarray(names).flatten()
    
    # 3. Now iteration yields pure strings (or numpy.str_), which str() handles perfectly
    surfaces = [surf_lookup(str(name)) for name in names]
    
    batch["y_af"] = jnp.stack(surfaces)
    return batch

data_dir = Path("/home/jordan/dev/datasets/mUIUC_5D.zarr")
data_dir.mkdir(parents=True, exist_ok=True)
run_dir = Path("/home/jordan/dev/EDEn/flowtangent/exp/XFOIL")

### Download Dataset (Optional)

In [ ]:
from huggingface_hub import hf_hub_download
import zipfile

zip_path = hf_hub_download(repo_id="eden-uiuc/mUIUC-5D", repo_type="dataset", filename="mUIUC_5D.zarr.zip")

if not (data_dir / 'zarr.json').exists():
    print("Extracting mUIUC 5D Dataset")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(data_dir)
    print("Extraction Complete.")
else:
    print ("Stored dataset detected.")

print(f"Dataset ready at: {data_dir}")

### Import Dataset

In [2]:
XFD_raw = ft.utils.data.XArrayDataset(data_dir).to_tabular(["run", "alpha"])
XFD = XFD_raw.dropna(["alpha"])
XFD._proxy._ds.load()

Dropping NaNs from dataset:   0%|          | 0/104 [00:00<?, ?it/s]

<xarray.Dataset> Size: 654MB
Dimensions:     (run: 358400, bend_var: 24, cond_var: 4, alpha: 42, polar_var: 9)
Coordinates:
  * bend_var    (bend_var) <U18 2kB 'Area' 'Slen' ... 'skin_Y_Ixx/t(Y-Yc)' 'J/t'
  * cond_var    (cond_var) <U10 160B 'flap_angle' 'flap_x/c' 'Re' 'Mach'
  * polar_var   (polar_var) <U7 252B 'alpha' 'CL' 'CD' ... 'Top_Itr' 'Bot_Itr'
Dimensions without coordinates: run, alpha
Data variables:
    bend_data   (run, bend_var) float32 34MB 0.0463 2.051 0.3786 ... 0.04929 nan
    conditions  (run, cond_var) float32 6MB -1.81 0.7673 ... 1.978e+05 0.5435
    foil_name   (run) <U50 72MB '2032c' '2032c' '2032c' ... 'ys930' 'ys930'
    polar_data  (run, alpha, polar_var) float32 542MB 0.0 0.5962 ... 5.937 86.99

In [3]:
unique_names = [str(n) for n in set(np.asarray(XFD_raw.foil_name))][1:]
surfs = [jnp.atleast_2d(surf_lookup(n)) for n in unique_names]
X_SURF = jnp.concatenate(surfs)

### Set up Parameters

In [4]:
# Inputs
y_af = Parameter(name="y_af", is_transform=True, transform_sources=("foil_name",))
Mach = Parameter(name="Mach")
Re = Parameter(name="Re")
flap_loc = Parameter(name="flap_x/c")
flap_angle = Parameter(name="flap_angle")
alpha = Parameter(name="alpha")

# Outputs
CL = Parameter(name="CL", is_output=True)
CD = Parameter(name="CD", is_output=True)
CM = Parameter(name="CM", is_output=True)

theta = (y_af, Mach, Re, flap_loc, flap_angle, alpha, CL, CD, CM)

### Set up Manifold

In [5]:
geofold = LinearPCA(physical_dim=256, latent_dim=16) # Airfoil Encoding
geofold = geofold.fit(X_SURF)

machfold    = ScalingManifold(method="normalize", mean=XFD.Mach.mean(), std=XFD.Mach.std())
refold      = ScalingManifold(method="normalize", mean=XFD.Re.mean(), std=XFD.Re.std())
hingefold   = ScalingManifold(method="normalize", mean=getattr(XFD, "flap_x/c").mean(), std=getattr(XFD, "flap_x/c").std())
deltafold   = ScalingManifold(method="normalize", mean=XFD.flap_angle.mean(), std=XFD.flap_angle.std())
alphafold   = ScalingManifold(method="normalize", mean=XFD.alpha.mean(), std=XFD.alpha.std())

mfold = geofold + machfold + refold + hingefold + deltafold + alphafold

## Manual Design Space

### Setup Design Space

In [12]:
XFDt = XFD.transform_variable(name2surf)

M=512
jitter=5e-3
complexity_weight=1.0

rng_key = jax.random.key(42)
rng_key, CL_key = jax.random.split(rng_key)

CL_surr = SVGPR(
    name="CL",
    key=CL_key,
    kernel=Matern52Kernel(21),
    input_dim=21, 
    output_dim=1,
    num_inducing=M,
    jitter=jitter,
    complexity_weight=complexity_weight,
    scale_mean=(XFD.CL.mean().item(),),
    scale_std=(XFD.CL.std().item(),),
)

rng_key, CD_key = jax.random.split(rng_key)

CD_surr = SVGPR(
    name="CD",
    key=CD_key,
    kernel=Matern52Kernel(21),
    input_dim=21, 
    output_dim=1,
    num_inducing=M,
    jitter=jitter,
    complexity_weight=complexity_weight,
    scale_mean=(XFD.CD.mean().item(),),
    scale_std=(XFD.CD.std().item(),),
)

rng_key, CM_key = jax.random.split(rng_key)

CM_surr = SVGPR(
    name="CM",
    key=CM_key,
    kernel=Matern52Kernel(21),
    input_dim=21, 
    output_dim=1,
    num_inducing=M,
    jitter=jitter,
    complexity_weight=complexity_weight,
    scale_mean=(XFD.CM.mean().item(),),
    scale_std=(XFD.CM.std().item(),),
)

surr = CL_surr + CD_surr + CM_surr

airspace = ActiveSpace(
    dataset=XFDt,
    manifold=mfold,
    surrogate=surr,
    parameters=theta,
)

In [ ]:
fit_DL, _ = airspace.get_dataloaders(batch_size=4096)
Z_batch, Y_batch = next(iter(ft.utils.data.LatentDataLoader(fit_DL, manifold=airspace.manifold)))

Z_batch_np = np.array(Z_batch)
Y_batch_norm = (Y_batch - Y_batch.mean(axis=0))/Y_batch.std(axis=0)

In [ ]:
kmeans = KMeans(n_clusters=M, n_init="auto", random_state=42)
kmeans.fit(Z_batch_np)
M_init = jnp.array(kmeans.cluster_centers_)

M_idxs = pairwise_distances_argmin(M_init, Z_batch_np)
mu_init = Y_batch_norm[M_idxs]

airspace = eqx.tree_at(
    lambda a: (
        a.surrogate.surrogates[0]._inducing_points,
        a.surrogate.surrogates[1]._inducing_points,
        a.surrogate.surrogates[2]._inducing_points,
    ),
    airspace,
    (M_init,) * 3
)

In [ ]:
    rng_key, test_key = jax.random.split(rng_key)
    ZDL, tZDL = airspace.get_dataloaders(batch_size=batch_size, test_split=0.1, split_key=test_key)
    ZDL = ft.utils.data.LatentDataLoader(ZDL, mfold_obj)
    tZDL = ft.utils.data.LatentDataLoader(tZDL, mfold_obj)
    lr = optax.cosine_decay_schedule(1e-2, 100 * len(ZDL) // batch_size, alpha=0.01) 
    airspace.surrogate.fit(data=ZDL, test_data=tZDL, learning_rate=lr, epochs=config["max_epochs"], stop_fn=NLPD_Plateau, logging_mode="off")

### Test Fit

In [ ]:
import jax.numpy as jnp
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import plotly.io as pio

def compute_and_plot_aerodynamic_complexity(Z, Y, num_bins=50, k_neighbors=5):
    """
    Diagnoses the fitness of a dataset for a stationary GP by computing 
    the Empirical Variogram and Local Lipschitz constants.
    
    Args:
        Z: array of shape (N, 16), the latent PCA coordinates.
        Y: array of shape (N, 1) or (N,), the aerodynamic targets (e.g., unnormalized C_L).
    """
    Z = jnp.atleast_2d(Z)
    Y = jnp.atleast_1d(Y).ravel()
    
    # ==========================================
    # 1. Compute Pairwise Distances Efficiently
    # ==========================================
    # L2 distance in latent space
    Z_sq = jnp.sum(Z**2, axis=1)
    dist_sq = Z_sq.reshape(-1, 1) + Z_sq.reshape(1, -1) - 2 * jnp.dot(Z, Z.T)
    Z_dist = jnp.sqrt(jnp.clip(dist_sq, min=1e-12))  # clip to avoid nan in sqrt
    
    # Squared differences in output space (Semivariance proxy)
    Y_diff_sq = (Y.reshape(-1, 1) - Y.reshape(1, -1)) ** 2
    
    # Absolute differences for Lipschitz gradients
    Y_diff_abs = jnp.abs(Y.reshape(-1, 1) - Y.reshape(1, -1))

    # ==========================================
    # 2. Compute Empirical Variogram (Semivariance)
    # ==========================================
    # Extract upper triangle to avoid redundant symmetric calculations
    triu_indices = jnp.triu_indices_from(Z_dist, k=1)
    dists_flat = np.array(Z_dist[triu_indices])
    gamma_flat = np.array(0.5 * Y_diff_sq[triu_indices])  # gamma(h) = 0.5 * (y_i - y_j)^2
    
    # Bin the distances
    max_dist = np.percentile(dists_flat, 98) # ignore extreme outliers for binning
    bins = np.linspace(0, max_dist, num_bins + 1)
    bin_centers = 0.5 * (bins[1:] + bins[:-1])
    
    # Calculate average semivariance in each distance bin
    bin_indices = np.digitize(dists_flat, bins)
    empirical_gamma = np.zeros(num_bins)
    
    for i in range(1, num_bins + 1):
        in_bin = (bin_indices == i)
        if np.any(in_bin):
            empirical_gamma[i-1] = np.mean(gamma_flat[in_bin])
        else:
            empirical_gamma[i-1] = np.nan

    # ==========================================
    # 3. Compute Local Lipschitz Constants (Gradients)
    # ==========================================
    # Mask out the self-distances
    mask = jnp.eye(Z_dist.shape[0], dtype=bool)
    Z_dist_no_self = jnp.where(mask, jnp.inf, Z_dist)
    
    # Find the indices of the k-nearest neighbors for every point
    nn_indices = jnp.argsort(Z_dist_no_self, axis=1)[:, :k_neighbors]
    
    # Calculate local gradient: |delta Y| / |delta Z| for nearest neighbors
    # Using advanced indexing to pluck the exact values
    row_indices = jnp.arange(Z_dist.shape[0])[:, None]
    local_deltaz = Z_dist[row_indices, nn_indices]
    local_deltay = Y_diff_abs[row_indices, nn_indices]
    
    lipschitz_vals = np.array(local_deltay / local_deltaz).flatten()

    # ==========================================
    # 4. Plotly Visualization Setup
    # ==========================================
    bw_pub = go.layout.Template(
        layout=go.Layout(
            plot_bgcolor="white", paper_bgcolor="white",
            font=dict(color="black", family="Serif", size=13),
            colorway=["black", "#444444", "#888888", "#bbbbbb"],
            colorscale=dict(sequential=px.colors.sequential.Greys),
            xaxis=dict(gridcolor="#eeeeee", zerolinecolor="black", linecolor="black", mirror=True, ticks="outside", showline=True),
            yaxis=dict(gridcolor="#eeeeee", zerolinecolor="black", linecolor="black", mirror=True, ticks="outside", showline=True),
        )
    )
    pio.templates.default = bw_pub

    fig = make_subplots(
        rows=1, cols=2, 
        subplot_titles=(
            "Empirical Variogram (GP Stationarity Check)", 
            f"Local Lipschitz Gradients (k={k_neighbors} Nearest Neighbors)"
        ),
        horizontal_spacing=0.1
    )

    # Panel 1: Variogram
    valid_bins = ~np.isnan(empirical_gamma)
    fig.add_trace(
        go.Scatter(
            x=bin_centers[valid_bins], y=empirical_gamma[valid_bins],
            mode="markers+lines",
            marker=dict(size=6, color="black"),
            line=dict(width=1.5, color="#888888")
        ), row=1, col=1
    )

    # Panel 2: Lipschitz Histogram
    # Clip the top 1% so massive outliers don't crush the x-axis binning
    clip_val = np.percentile(lipschitz_vals, 99)
    fig.add_trace(
        go.Histogram(
            x=np.clip(lipschitz_vals, min=0, max=clip_val),
            nbinsx=50,
            marker=dict(color="#cccccc", line=dict(color="black", width=1))
        ), row=1, col=2
    )

    # Add vertical line for the 95th percentile gradient
    p95_lip = np.percentile(lipschitz_vals, 95)
    fig.add_vline(
        x=p95_lip, line_dash="dash", line_color="black", line_width=2,
        annotation_text=f"95th Pctl:<br>{p95_lip:.2f}", 
        annotation_position="top right",
        row=1, col=2
    )

    # Formatting
    fig.update_xaxes(title_text="Latent Distance $||Z_i - Z_j||_2$", row=1, col=1)
    fig.update_yaxes(title_text=r"Semivariance \gamma(h)", row=1, col=1)
    
    fig.update_xaxes(title_text="Gradient $|DC_L| / ||DZ||_2$", row=1, col=2)
    fig.update_yaxes(title_text="Frequency", row=1, col=2)

    fig.update_layout(
        height=450, width=1000,
        margin=dict(l=60, r=40, t=60, b=60),
        showlegend=False
    )

    fig.show()

In [ ]:
space_DL, _ = airspace.get_dataloaders(batch_size=4096)
Z_batch, Y_batch = next(iter(ft.utils.data.LatentDataLoader(space_DL, manifold=airspace.manifold)))
compute_and_plot_aerodynamic_complexity(Z_batch, Y_batch[:, 0])


In [ ]:
import jax.numpy as jnp
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

def plot_worst_aerodynamic_cliffs(Z_batch, Y_batch, basis, mu, top_n=3):
    """
    Finds the pairs of points with the highest local gradients and plots 
    their physical geometries side-by-side to diagnose data corruption vs physics.
    
    Args:
        Z_batch: (N, D) latent inputs (must include geometry; if AoA is appended, adjust slicing)
        Y_batch: (N,) target C_L values
        basis: (16, N_coords) PCA basis to decode geometry
        mu: (N_coords,) PCA mean
    """
    Z = jnp.atleast_2d(Z_batch)
    Y = jnp.atleast_1d(Y_batch).ravel()
    
    # 1. Compute Gradients
    Z_sq = jnp.sum(Z**2, axis=1)
    dist_sq = Z_sq.reshape(-1, 1) + Z_sq.reshape(1, -1) - 2 * jnp.dot(Z, Z.T)
    Z_dist = jnp.sqrt(jnp.clip(dist_sq, a_min=1e-12))
    
    Y_diff_abs = jnp.abs(Y.reshape(-1, 1) - Y.reshape(1, -1))
    
    # Mask out self-distances and the lower triangle to avoid duplicates
    mask = jnp.triu(jnp.ones_like(Z_dist, dtype=bool), k=1)
    gradients = jnp.where(mask, Y_diff_abs / Z_dist, -1.0)
    
    # 2. Find the top N offending pairs
    flat_indices = jnp.argsort(gradients.flatten())[::-1][:top_n]
    row_idx, col_idx = jnp.unravel_index(flat_indices, gradients.shape)
    
    # Setup Plotly
    bw_pub = go.layout.Template(
        layout=go.Layout(
            plot_bgcolor="white", paper_bgcolor="white",
            font=dict(color="black", family="Serif", size=12),
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, scaleanchor="x", scaleratio=1),
        )
    )
    pio.templates.default = bw_pub

    fig = make_subplots(
        rows=top_n, cols=1, 
        subplot_titles=[f"Cliff {i+1}" for i in range(top_n)],
        vertical_spacing=0.1
    )

    half_idx = mu.shape[0] // 2
    x_coords = np.linspace(0, 1, half_idx)
    x_wrap = np.concatenate([x_coords, x_coords[::-1]])

    for i in range(top_n):
        idx_A, idx_B = int(row_idx[i]), int(col_idx[i])
        
        # If your Z vector has AoA/Re appended (e.g. 17D or 18D), slice only the first 16 for geometry
        geom_A = Z[idx_A, :16]
        geom_B = Z[idx_B, :16]
        
        # Decode back to physical space
        X_A = np.dot(geom_A, basis) + mu
        X_B = np.dot(geom_B, basis) + mu
        
        y_wrap_A = np.concatenate([X_A[:half_idx], X_A[half_idx:][::-1]])
        y_wrap_B = np.concatenate([X_B[:half_idx], X_B[half_idx:][::-1]])

        # Plot Airfoil A
        fig.add_trace(go.Scatter(
            x=x_wrap, y=y_wrap_A, mode="lines",
            line=dict(color="black", width=2), name=f"Airfoil A (C_L: {Y[idx_A]:.3f})"
        ), row=i+1, col=1)

        # Plot Airfoil B (dashed, red to highlight the difference)
        fig.add_trace(go.Scatter(
            x=x_wrap, y=y_wrap_B, mode="lines",
            line=dict(color="#d62728", width=2, dash="dash"), name=f"Airfoil B (C_L: {Y[idx_B]:.3f})"
        ), row=i+1, col=1)
        
        # Add annotation detailing the cliff
        dist_val = Z_dist[idx_A, idx_B]
        grad_val = gradients[idx_A, idx_B]
        fig.add_annotation(
            x=0.5, y=-0.15,
            text=f"ΔLatent: {dist_val:.2e} | ΔC_L: {Y_diff_abs[idx_A, idx_B]:.3f} | Gradient: {grad_val:.2e}",
            showarrow=False, font=dict(size=14), row=i+1, col=1
        )

    fig.update_layout(
        title=dict(text="Top Gradient Edge Cases (Overlaid Geometries)", x=0.5),
        height=300 * top_n, width=800,
        showlegend=True
    )
    
    fig.show()

# Run it using the batch and your saved PCA basis/mu
# plot_worst_aerodynamic_cliffs(X_batch, Y_batch_CL, pca_basis, pca_mu, top_n=3)

### Diagnostic Setup

In [ ]:
def generate_diagnostics(model, test_x, test_y, n_bins=21):
    # 1. Base Predictions & Metrics
    preds = jax.vmap(model.surrogate.predict)(test_x)
    residuals = preds.means - test_y
    absolute_errors = jnp.abs(residuals)

    safe_y = jnp.where(jnp.abs(test_y) < 1e-4, 1e-4, jnp.abs(test_y))
    
    pred_std = jnp.sqrt(preds.variances)
    z_scores = residuals / pred_std

    metrics = {
        metr: {
            "MAE": jnp.mean(absolute_errors[:, i]).item(),
            "RMSE": jnp.sqrt(jnp.mean(residuals[:, i]**2)).item(),
            "Z_Score_Variance": jnp.var(z_scores[:, i]).item() 
        }
        for i, metr in enumerate(["CL", "CD", "CM"])
    }

    # 2. Vectorized Binning
    m_dist = jnp.linalg.norm(test_x[:, :16], axis=1)
    
    bins_dict = {
        "Geometry": jnp.linspace(0., jnp.max(m_dist), n_bins),
        "Mach": jnp.linspace(0.0, 0.6, n_bins),
        "Re": jnp.logspace(np.log10(50_000), np.log10(3_000_000), n_bins),
        "Pivot": jnp.linspace(0.6, 0.85, n_bins),
        "Angle": jnp.linspace(-10, 15, n_bins),
        "Alpha": jnp.linspace(-5, 15, n_bins)
    }

    test_x_hat = jax.vmap(model.reconstruct)(test_x)
    features_dict = {
        "Geometry": m_dist,
        "Mach": test_x_hat["Mach"].flatten(),
        "Re": test_x_hat["Re"].flatten(),
        "Pivot": test_x_hat["flap_x/c"].flatten(),
        "Angle": test_x_hat["flap_angle"].flatten(),
        "Alpha": test_x_hat["alpha"].flatten(),
    }
    
    param_names = list(bins_dict.keys())
    features = jnp.stack([features_dict[p] for p in param_names]) # (6_params, N_points)
    bins_stack = jnp.stack([bins_dict[p] for p in param_names])   # (6_params, n_bins)
    
    # Digitize all features at once. Indices 1 to (n_bins-1) correspond to our defined intervals.
    idxs = jax.vmap(lambda f, b: jnp.digitize(f, b, right=True))(features, bins_stack)
    
    # Create boolean mask: (6_params, N_points, n_bins + 1)
    n_indices = n_bins + 1
    masks = idxs[..., None] == jnp.arange(n_indices)
    
    # 3. Tensor Aggregation for Binned Means
    sq_frac_errs = ((residuals / safe_y) ** 2).T  # (3_metrics, N_points)
    
    # Sum errors across points in each bin: (3_metrics, 6_params, n_indices)
    sum_sq_frac = jnp.einsum('mn,pnb->mpb', sq_frac_errs, masks)
    
    # Count points in each bin: (6_params, n_indices)
    counts = jnp.sum(masks, axis=1)
    
    # Divide by count (MSE), then square root (RMSE)
    mean_sq_frac = jnp.where(counts > 0, sum_sq_frac / counts, np.nan)
    rms_frac_errors = jnp.sqrt(mean_sq_frac)
    
    valid_rms_frac = rms_frac_errors[:, :, 1:n_bins]
    
    errors = {}
    for p_idx, param in enumerate(param_names):
        b = bins_dict[param]
        centers = (b[:-1] + b[1:]) / 2.0
        
        errors[param] = {
            "bins": b,
            "centers": centers,
            "means": { 
                metr: valid_rms_frac[m_idx, p_idx, :]
                for m_idx, metr in enumerate(["CL", "CD", "CM"])
            }
        }
        
    return preds, metrics, z_scores, errors

In [ ]:
from plotly import graph_objects as go
from plotly.subplots import make_subplots

def plot_error_matrices(errors):
    param_names = list(errors.keys())
    metrics = ["CL", "CD", "CM"]
    
    fig = make_subplots(
        rows=2, cols=3, 
        subplot_titles=param_names,
        horizontal_spacing=0.08,
        vertical_spacing=0.15
    )
    
    for i, param in enumerate(param_names):
        row = (i // 3) + 1
        col = (i % 3) + 1
        
        centers = errors[param]["centers"]
        z_data = [errors[param]["means"][m] for m in metrics]
        
        fig.add_trace(
            go.Heatmap(
                z=z_data,
                x=centers,
                y=metrics,
                colorscale="Viridis", 
                zmin=0.0, 
                zmax=2.0,
                # Optional: set zmax=0.5 (50%) to prevent zero-crossing spikes from washing out the color scale
                showscale=(i == 0),
                colorbar=dict(
                    title="RMS % Error",
                    tickformat=".1%"  # Automatically formats 0.15 as 15.0%
                ) if i == 0 else None,
            ),
            row=row, col=col
        )
        
        if param == "Re":
            fig.update_xaxes(type="log", row=row, col=col)
            
    fig.update_layout(
        title_text="Surrogate RMS Percentage Error",
        height=600, 
        width=1000,
        plot_bgcolor='white'
    )
    
    return fig

### Diagnostics

In [ ]:
# 1. Generate globally random indices
total_points = len(airspace.dataset)
test_indices = np.random.choice(total_points, size=8192, replace=False)

# Sort indices to ensure Zarr reads the chunks sequentially without thrashing
test_indices.sort()

# 2. Extract the batch instantly via the new vectorized fast-path
raw_batch = airspace.dataset[test_indices]

geo_x = raw_batch["y_af"]

# 3. Stack the inputs and targets
scalar_x = jnp.stack([
    raw_batch["Mach"], 
    raw_batch["Re"], 
    raw_batch["flap_x/c"], 
    raw_batch["flap_angle"], 
    raw_batch["alpha"]
], axis=-1)

raw_x = jnp.concatenate([geo_x, scalar_x], axis=-1)

test_y = jnp.stack([
    raw_batch["CL"], 
    raw_batch["CD"], 
    raw_batch["CM"]
], axis=-1)

# 4. Push through the manifold
test_x = jax.vmap(airspace.manifold.encode)(raw_x)

# 5. Generate matrices
preds, metrics, z_scores, errors = generate_diagnostics(airspace, test_x, test_y)


In [ ]:
plot_error_matrices(errors)

## Hyperparameter Search

### Setup

In [10]:
from collections import deque, defaultdict
import tempfile
import equinox as eqx
import json
import warnings

import ray
from ray import tune
from ray.tune import CLIReporter, JupyterNotebookReporter
from ray.tune.schedulers import ASHAScheduler
from ray.tune.logger.tensorboardx import TBXLoggerCallback
from filelock import FileLock

batch_size = 8192

def train_svgp_trial(config, XFD_data, mfold_obj, theta_params):
    # Unpack hyperparams
    M = config["num_inducing"]
    jitter = config["jitter"]

    @lru_cache(maxsize=2048)
    def local_surf_lookup(name: str):
        safe_name = name.replace("-", "_").replace(".", "_")
        if safe_name and safe_name[0] in string.digits:
            safe_name = f"af_{safe_name}"

        af = getattr(ft.airfoils, safe_name)

        y_af = jnp.concatenate([af.y_upper, af.y_lower])
        return y_af

    def local_name2surf(batch: dict):
        names = batch["foil_name"]
        
        if hasattr(names, "values"):
            names = names.values
            
        names = np.asarray(names).flatten()
        
        surfaces = [local_surf_lookup(str(name)) for name in names]
        
        batch["y_af"] = jnp.stack(surfaces)
        return batch

    XFD_data = XFD_data.transform_variable(local_name2surf)
    rng_key = jax.random.key(42)
    rng_key, CL_key = jax.random.split(rng_key)
    
    # 3. Initialize Model
    CL_surr = SVGPR(
        name="CL",
        key=CL_key,
        kernel=Matern52Kernel(21),
        input_dim=21, 
        output_dim=1,
        num_inducing=M,
        jitter=jitter,
        scale_mean=(XFD_data.CL.mean().item(),),
        scale_std=(XFD_data.CL.std().item(),),
    )

    rng_key, CD_key = jax.random.split(rng_key)

    CD_surr = SVGPR(
        name="CD",
        key=CD_key,
        kernel=Matern52Kernel(21),
        input_dim=21, 
        output_dim=1,
        num_inducing=M,
        jitter=jitter,
        scale_mean=(XFD_data.CD.mean().item(),),
        scale_std=(XFD_data.CD.std().item(),),
    )

    rng_key, CM_key = jax.random.split(rng_key)

    CM_surr = SVGPR(
        name="CM",
        key=CM_key,
        kernel=Matern52Kernel(21),
        input_dim=21, 
        output_dim=1,
        num_inducing=M,
        jitter=jitter,
        scale_mean=(XFD_data.CM.mean().item(),),
        scale_std=(XFD_data.CM.std().item(),),
    )

    surr = CL_surr + CD_surr + CM_surr

    airspace = ActiveSpace(
        dataset=XFD_data,
        manifold=mfold_obj,
        surrogate=surr,
        parameters=theta_params,
    )

    # Fit K-means inducing points and update surrogates
    fit_DL, _ = airspace.get_dataloaders(batch_size=batch_size)
    Z_batch, Y_batch = next(iter(ft.utils.data.LatentDataLoader(fit_DL, manifold=airspace.manifold)))
    
    Z_batch_np = np.array(Z_batch)
    
    kmeans = KMeans(n_clusters=M, n_init="auto", random_state=42)
    kmeans.fit(Z_batch_np)
    M_init = jnp.array(kmeans.cluster_centers_)

    airspace = eqx.tree_at(
        lambda a: (
            a.surrogate.surrogates[0]._inducing_points,
            a.surrogate.surrogates[1]._inducing_points,
            a.surrogate.surrogates[2]._inducing_points,
        ),
        airspace,
        (M_init,) * 3
    )

    leaderboard_dir = run_dir / "best_models"
    leaderboard_dir.mkdir(exist_ok=True)
    leaderboard = leaderboard_dir / "leaderboard.json"
    lock_path = leaderboard_dir / "leaderboard.lock"

    if not leaderboard.exists():
        with open(leaderboard, "w") as f:
            json.dump({}, f)

    best_nlpds = defaultdict(lambda: float('inf'))
    wait_epochs = 0
    surrogate_key = None

    def NLPD_Plateau(metrics, current_model):
        # Report to Ray so ASHA can rank and kill trials
        nonlocal best_nlpds, wait_epochs, surrogate_key
        
        default_mae = np.zeros(1)
        NLPD_key = [k for k in metrics.keys() if "NLPD" in k][0]
        MAE_key = [k for k in metrics.keys() if "MAE" in k][0]
        KL_key = [k for k in metrics.keys() if "KL" in k][0]

        if surrogate_key != NLPD_key:
            surrogate_key = NLPD_key
            wait_epochs = 0
        
        clean_metrics = {
            NLPD_key: float(metrics.get(NLPD_key, 0.0)) / len(XFD_data),
            MAE_key: float(metrics.get(MAE_key, default_mae)[0]),
            KL_key: float(metrics.get(KL_key, 0)),
            "epoch": metrics.get("epoch", 0) + 1
        }
        
        # Implement manual plateau stopping
        current_nlpd = clean_metrics[NLPD_key]
        current_epoch = clean_metrics["epoch"]
        checkpoint = None
        
        is_new_local_best = current_nlpd < best_nlpds[NLPD_key]

        if is_new_local_best:
            best_nlpds[NLPD_key] = current_nlpd
            wait_epochs = 0
        else:
            wait_epochs += 1

        # Wait a set number epochs and check if we've waited long enough to terminate
        if current_epoch >= 50 and (is_new_local_best or current_epoch % 50 == 0):
            with tempfile.TemporaryDirectory() as temp_dir:
                model_path = Path(temp_dir) / "resume_state.eqx"
                eqx.tree_serialise_leaves(model_path, current_model)
                checkpoint = tune.Checkpoint.from_directory(temp_dir)
                tune.report(clean_metrics, checkpoint=checkpoint)
            try:
                with FileLock(lock_path):
                    with open(leaderboard, "r") as f:
                        global_bests = json.load(f)

                    global_best_nlpd = global_bests.get(NLPD_key, float('inf'))

                    if current_nlpd < global_best_nlpd:
                        global_bests[NLPD_key] = current_nlpd
                        with open(leaderboard, "w") as f:
                            json.dump(global_bests, f)

                        leader = leaderboard_dir / f"{NLPD_key}.eqx"
                        eqx.tree_serialise_leaves(leader, current_model)
            except:
                pass
                
        else:
            tune.report(clean_metrics)
        
        if current_epoch >= 50 and wait_epochs >= 25:
            return True  # Terminate early
                
        return False  # Continue training

    rng_key, test_key = jax.random.split(rng_key)
    ZDL, tZDL = airspace.get_dataloaders(batch_size=batch_size, test_split=0.1, split_key=test_key)
    ZDL = ft.utils.data.LatentDataLoader(ZDL, mfold_obj)
    tZDL = ft.utils.data.LatentDataLoader(tZDL, mfold_obj)
    lr = optax.cosine_decay_schedule(1e-2, 100 * len(ZDL) // batch_size, alpha=0.01) 
    airspace.surrogate.fit(data=ZDL, test_data=tZDL, learning_rate=lr, epochs=config["max_epochs"], stop_fn=NLPD_Plateau, logging_mode="off")

def run_hyperparameter_sweep():
    # Define the search space
    search_space = {
        "num_inducing": tune.grid_search([256, 512, 1024, 2048]),
        "jitter": tune.loguniform(5e-4, 1e-1),
        # "complexity_weight": tune.loguniform(1e-2, 10.0),
        "max_epochs": 100 
    }
    
    trainable_with_params = tune.with_parameters(
        train_svgp_trial,
        XFD_data=XFD,
        mfold_obj=mfold,
        theta_params=theta,
    )

    tuner = tune.Tuner(
        tune.with_resources(trainable_with_params, resources={"cpu": 4, "gpu": 1.0}),
        param_space=search_space,
        tune_config=tune.TuneConfig(
            num_samples=3,      
            # scheduler=scheduler,
        ),
        run_config=tune.RunConfig(
            name="svgp_stage_5",
            callbacks=[TBXLoggerCallback()],
            checkpoint_config=tune.CheckpointConfig(num_to_keep=1)
        )
    )
    
    try:
        results = tuner.fit()
    except KeyboardInterrupt:
        print("\nSweep interrupted by user. Shutting down Ray cleanly...")
    finally:
        ray.shutdown()

### Ray Initialization

In [8]:
import subprocess

def start_ray_metrics():
    """Kills stale metrics servers and boots fresh ones attached to the current Ray session."""
    # 1. Quietly kill any old instances from previous runs
    subprocess.run(
        ["killall", "prometheus", "grafana-server"], 
        stdout=subprocess.DEVNULL, 
        stderr=subprocess.DEVNULL
    )
    
    # 2. Start Prometheus in the background
    subprocess.Popen([
        "prometheus", 
        "--config.file=/tmp/ray/session_latest/metrics/prometheus/prometheus.yml"
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    
    # 3. Start Grafana in the background with our custom tmp paths
    subprocess.Popen([
        "grafana-server", 
        "--config", "/tmp/ray/session_latest/metrics/grafana/grafana.ini",
        "--homepath", "/usr/share/grafana",
        "cfg:default.paths.data=/tmp/grafana-data",
        "cfg:default.paths.logs=/tmp/grafana-logs",
        "web"
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

if ray.is_initialized():
    ray.shutdown()
context = ray.init(include_dashboard=True, dashboard_port=8265, ignore_reinit_error=True)
start_ray_metrics()

2026-11-18 17:21:11,509	WARNING authentication_token_setup.py:85 -- Token authentication is enabled for this Ray cluster. Set RAY_AUTH_MODE=disabled to opt out. For more information, see https://docs.ray.io/en/latest/ray-security/token-auth.html
/usr/lib/python3.12/subprocess.py:1885: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = _fork_exec(
/usr/lib/python3.12/subprocess.py:1885: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = _fork_exec(
2026-11-18 17:21:17,184	INFO worker.py:2029 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 


### Execution

In [11]:
run_hyperparameter_sweep()

2026-11-20 11:10:29,974	WARNING tune.py:219 -- Stop signal received (e.g. via SIGINT/Ctrl+C), ending Ray Tune run. This will try to checkpoint the experiment state one last time. Press CTRL+C (or send SIGINT/SIGKILL/SIGTERM) to skip. 
2026-11-20 11:10:29,980	INFO tune.py:1007 -- Wrote the latest version of all result files and experiment state to '/home/jordan/ray_results/svgp_stage_5' in 0.0045s.
2026-11-20 11:10:40,005	INFO tune.py:1039 -- Total run time: 4802.50 seconds (4792.43 seconds for the tuning loop).
2026-11-20 11:10:40,006	WARNING tune.py:1054 -- Experiment has been interrupted, but the most recent state was saved.
Resume experiment with: Tuner.restore(path="/home/jordan/ray_results/svgp_stage_5", trainable=...)
2026-11-20 11:10:40,020	WARNING experiment_analysis.py:186 -- Failed to fetch metrics for 10 trial(s):
- train_svgp_trial_21356_00002: FileNotFoundError('Could not fetch metrics for train_svgp_trial_21356_00002: both result.json and progress.csv were not found at /h